In [ ]:
import sys

!which {sys.executable}

In [ ]:
import sys
sys.path.insert(0, '..')

In [ ]:
# %load /ut3/jerome/github/research/project/wakai/notebooks/init_ipynb
import os
import sys
import matplotlib.pyplot as pl

# sys.path.insert(0, '../code')
# assert os.environ["CONDA_DEFAULT_ENV"] == 'wakai'

plotpar = {
           'font.size': 20,
           'xtick.labelsize': 16,
           'ytick.labelsize': 16,
           'xtick.direction': 'in',
           'ytick.direction': 'in',
           'xtick.color':'black',
           'ytick.color':'black',
           'xtick.major.width':2,
           'ytick.major.width':2,
           'xtick.major.size':10,
           'ytick.major.size':10,
           'xtick.minor.width':1,
           'ytick.minor.width':1,
           'xtick.minor.size':6,
           'ytick.minor.size':6,
           'axes.labelsize': 16,
           'axes.labelcolor':'black',
           'axes.labelcolor':'black',
           'axes.spines.top':True,
           'axes.spines.right':True,
           'axes.linewidth':3,
           'axes.edgecolor':'black',
           'figure.facecolor':'none',
           'legend.facecolor':'none',
           'legend.fontsize': 16,
           'text.color':'black',
           'pdf.fonttype': 42,
           #'text.usetex': False,
           }
pl.rcParams.update(plotpar)

# Moranta+2022

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Moranta2022", catalog_type="cluster")
tabs = cd.get_tables_from_vizier(row_limit=-1, save=True)

In [ ]:
df = tabs[1].to_pandas()
df

In [ ]:
df_mem = tabs[0].to_pandas()
df_mem

In [ ]:
gaiaid = 3762515188088861184
df_mem.Gaia.isin([gaiaid]).sum()

In [ ]:
idx=df_mem.Gaia==gaiaid
criusid = df_mem[idx].squeeze().Crius
criusid

In [ ]:
df_mem[df_mem.Crius==criusid].to_csv('temp.csv', index=False)

# Perren+2023 catalog

https://zenodo.org/records/19488683

```
Name(s)          : Name(s) associated to the OC
name             : Formatted naming to match the members file 'main name'
N_50             : Number of estimated members with P>0.5
r_50             : Radius that contains half the members (in arcmin)
RA_ICRS          : Right ascension
DE_ICRS          : Declination
GLON             : Galactic longitude
GLAT             : Galactic latitude
Plx              : Parallax [mas]
pmRA             : Right ascension proper motion [mas/yr]
pmDE             : Declination proper motion [mas/yr]
Rv               : Radial velocity [km/s]
N_Rv             : Number of members used to estimate the radial velocity
Dist_[kpc]       : Distance [kpc]
Dist_STDDEV      : Distance standard deviation
Av_[mag]         : Absorption in the V band [mag]
Av_STDDEV]       : Absorption standard deviation
Diff_ext_[mag]   : Differential extinction [mag]
Diff_ext_STDDEV  : Differential extinction standard deviation
Age_[Myr]        : Age [Myr]
Age_STDDEV       : Age standard deviation
FeH_[dex]        : Metallicity [dex]
FeH_STDDEV       : Metallicity standard deviation
Mass_[Msun]      : Mass [Msun]
Mass_STDDEV      : Mass standard deviation
Binary_fr        : Estimated binary fraction
Binary_fr_STDDEV : Estimated binary fraction standard deviation
BSS              : Blue stragglers (total number)
BSS_STDDEV       : Blue stragglers standard deviation
C3               : Combined quality class (see Perren et al. 2023)
P_dup            : Probability that this entry is a duplicate of a previous one
UTI              : UCC Trust Index, a measure of the reliability of the cluster
bad_oc           : Flag that indicates that this is likely an asterism,
                   moving group or artifact (based on UTI value, duplication
                   probability and presence in literature)
```

In [ ]:
import pandas as pd

df = pd.read_csv('../data/Perren2023/UCC_cat.csv')
df.tail()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
# non-negative parallax
df = df.dropna(subset=['RA_ICRS','DE_ICRS','Plx'])[df.Plx>0]
df.shape

In [ ]:
df['Name(s)'].unique()

In [ ]:
df.name.unique().shape

In [ ]:
df['Name(s)'].str.split(';').explode().dropna().unique().shape

In [ ]:
df_mem = pd.read_parquet('../data/Perren2023/UCC_members.parquet')
df_mem.tail()

In [ ]:
df_mem.shape

In [ ]:
df_mem.columns

In [ ]:
gid = 1079346279581047296
# gid = 5239758155778687360 #test
df_mem.Source.isin([gid]).sum()

In [ ]:
#remove spaces
df['Name'] = df.ID.apply(lambda x: x.lower().replace(' ', '').split(';')[0])
df.Name.isin(df_mem.Name).sum()

In [ ]:
len(df_mem.Name.unique())

https://ucc.ar/_clusters/ngc1039/

https://ucc.ar/_clusters/melotte20/

In [ ]:
%matplotlib inline

from wakai.catalog import plot_xyz_uvw

# name = "Melotte 102"
name = "Melotte 20" #alpha Per
cname = name.lower().replace(' ', '')

d = df_mem.query("Name==@cname")
assert len(d)>0
assert sum(d.Prob>0.9)>10
fig, axs = plot_xyz_uvw(d, color='Prob', 
                       # target_gaiaid=gaiaid, 
                       # target_label=f"TOI-{toiid}\nProb={prob*100:.1f}%", 
                       # target_color="r",
                       # target_size=500,
                       marker='.',
                       cmap="RdBu",
                       frame='galactic',
                       use_LSR=False
                     );
fig.suptitle(name)
axs = axs.flatten()
ulims = -20,20
vlims = -20,20
wlims = 0, 10
# axs[3].set_xlim(*ulims); axs[3].set_ylim(*vlims)
# axs[4].set_xlim(*vlims); axs[4].set_ylim(*wlims)
# axs[5].set_xlim(*wlims); axs[5].set_ylim(*ulims)

In [ ]:
df.sort_values(by="Plx", ascending=False).iloc[10:15]

### Interactive

In [ ]:
from wakai.catalog import get_transformed_coord

d = get_transformed_coord(d)
d

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = 'iframe' 

fig = go.Figure(data=[
    go.Scatter3d(
        x=d['X'], y=d['Y'], z=d['Z'],
        mode='markers',
        marker=dict(size=1, color=d['Gmag'], colorscale='Viridis')
    )
])
# fig.update_layout(scene=dict(aspectmode='data'))
fig

# Hunt+2024 catalog

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Hunt2024", catalog_type="cluster")
tabs = cd.get_tables_from_vizier(row_limit=-1, save=True)

In [ ]:
df = tabs[0].to_pandas()
df_mem = tabs[1].to_pandas()
df_mem.head()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({#"Cluster": "Name",
                        # "Source": "GaiaDR3",
                        # "Proba": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem.head()

In [ ]:
%matplotlib inline

from wakai.catalog import plot_xyz_uvw

cname = "Alessi_37"
d = df_mem.query("Name==@cname")
assert len(d)>0
assert sum(d.Prob>0.5)>10
fig = plot_xyz_uvw(d, 
                   color='Prob', 
                   # target_gaiaid=gaiaid, 
                   # target_label=f"TOI-{toiid}\nProb={prob*100:.1f}%", 
                   # target_color="r",
                   # target_size=500,
                   marker='.',
                   cmap="viridis");
fig.suptitle(cname)

# Cantat-Gaudin+2020 catalog

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="CantatGaudin2020", catalog_type="cluster")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
df = tabs[0].to_pandas()
df_mem = tabs[1].to_pandas()
df_mem.head()

In [ ]:
#replace name to be compatible with other catalogs' naming formats
df_mem = df_mem.rename({"Cluster": "Name",
                        "Source": "GaiaDR3",
                        "Proba": "Prob",
                        "RA_ICRS": "ra",
                        "DE_ICRS": "dec",
                        "pmRA": "pmra",
                        "pmDE": "pmdec",
                        "Plx": "parallax",
                        "RV": "radial_velocity"
                        }, axis=1)
df_mem.head()

In [ ]:
%matplotlib inline

from wakai.catalog import plot_xyz_uvw

cname = "NGC_1039"
d = df_mem.query("Name==@cname")
assert len(d)>0
assert sum(d.Prob>0.5)>10
fig = plot_xyz_uvw(d, 
                   color='Prob', 
                   # target_gaiaid=gaiaid, 
                   # target_label=f"TOI-{toiid}\nProb={prob*100:.1f}%", 
                   # target_color="r",
                   # target_size=500,
                   marker='.',
                   cmap="viridis");
fig.suptitle(cname)

# Plots

In [ ]:
from wakai.catalog import plot_cmd

cname = "NGC_1039"
d = df_mem.query("Name==@cname")
ax = plot_cmd(d, xaxis='BP-RP', yaxis='Gmag', color='BP-RP')
ax.set_title(cname)

# Custom cluster catalog

https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/A+A/677/A162

In [ ]:
from wakai.catalog import VIZIER_KEYS_CLUSTER_CATALOG

#add custom catalog for Hyades cluster
VIZIER_KEYS_CLUSTER_CATALOG['Brandner2023a'] = "J/AJ/165/108"
VIZIER_KEYS_CLUSTER_CATALOG['Brandner2023b'] = "J/A+A/677/A162"

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Brandner2023a", catalog_type="cluster")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
df_hyades = tabs[0].to_pandas()
df_hyades.head()

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Brandner2023b", catalog_type="cluster")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
df_pleiades = tabs[0].to_pandas()
df_pleiades.head()

In [ ]:
df.columns

In [ ]:
# fig = pl.figure()
# ax = pl.subplot(111, projection="mollweide")
ax = df_hyades.plot.scatter(x='RA_ICRS', y='DE_ICRS', label='Hyades', color='C0')
_ = df_pleiades.plot.scatter(x='RA_ICRS', y='DE_ICRS', label='Pleiades', color='C1', ax=ax)
ax.plot(80.066927, -4.239248, 'ro', label='TOI 4364')
ax.legend()

In [ ]:
toi4364_gaiaid = 3210444215030339584
df_hyades.Gaia.isin([3210444215030339584]).sum()

# Custom lithium catalog

https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/ApJS/271/58

In [ ]:
from wakai.catalog import VIZIER_KEYS_LiEW_CATALOG

#add custom catalog for Lithium
VIZIER_KEYS_LiEW_CATALOG['Ding2024'] = "J/ApJS/271/58"

In [ ]:
from wakai.catalog import CatalogDownloader

cd = CatalogDownloader(catalog_name="Ding2024", catalog_type="liew")
tabs = cd.get_tables_from_vizier(row_limit=-1)

In [ ]:
df_li = tabs[0].to_pandas()
df_li.head()

In [ ]:
df_li.shape

# FOV

In [ ]:
from astropy import units as u
from astropy.coordinates import SkyCoord
from astroplan.plots import plot_finder_image

fov_rad = 60*u.arcmin
obj_coord = SkyCoord(ra=ra, dec=dec, unit=(u.hourangle, u.deg))

#plot fov
ax, hdu = plot_finder_image(obj_coord, fov_radius=fov_rad, survey='DSS2 Red', reticle=True)